<a href="https://colab.research.google.com/github/ajaykumar080286/MachineLearning/blob/main/optuna_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
#!pip install optuna
import pandas as pd
import numpy as np

import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier


In [65]:
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']


In [66]:
df=pd.read_csv(url,names=columns)

In [67]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [68]:
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)
df.fillna(df.mean(), inplace=True)

In [69]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.00000,180.000000,32.9,0.171,63,0
764,2,122.0,70.0,27.00000,155.548223,36.8,0.340,27,0
765,5,121.0,72.0,23.00000,112.000000,26.2,0.245,30,0
766,1,126.0,60.0,29.15342,155.548223,30.1,0.349,47,1


In [70]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.00000,180.000000,32.9,0.171,63,0
764,2,122.0,70.0,27.00000,155.548223,36.8,0.340,27,0
765,5,121.0,72.0,23.00000,112.000000,26.2,0.245,30,0
766,1,126.0,60.0,29.15342,155.548223,30.1,0.349,47,1


In [71]:
X=df.drop('Outcome',axis=1)
y=df['Outcome']

In [72]:
X_train, X_test, y_train, y_test= train_test_split(X,y ,test_size=0.2, random_state=42)

In [73]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [74]:
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (614, 8)
Test set shape: (154, 8)


In [83]:
# Define the objective function
def objectiveFunction(trial):
  # Suggest values for the hyperparameters
  n_estimators=trial.suggest_int('n_estimators',50,200)
  max_depth=trial.suggest_int('max_depth',3,20)

   # Create the RandomForestClassifier with suggested hyperparameters

  model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)

  # Perform 3-fold cross-validation and calculate accuracy

  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score  # Return the accuracy score for Optuna to maximize

In [ ]:
# Create a study object and optimize the objective function
study=optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objectiveFunction,n_trials=50)

[I 2025-12-19 12:33:25,213] A new study created in memory with name: no-name-469d916a-8bd3-45d4-b406-7f7a354a6c45
[I 2025-12-19 12:33:27,012] Trial 0 finished with value: 0.7784791965566714 and parameters: {'n_estimators': 174, 'max_depth': 5}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:27,686] Trial 1 finished with value: 0.7670891120675912 and parameters: {'n_estimators': 70, 'max_depth': 5}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:29,071] Trial 2 finished with value: 0.7540490993145226 and parameters: {'n_estimators': 164, 'max_depth': 3}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:29,680] Trial 3 finished with value: 0.755706998246453 and parameters: {'n_estimators': 61, 'max_depth': 9}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:31,104] Trial 4 finished with value: 0.780113183484776 and parameters: {'n_estimators': 140, 'max_depth': 14}. Best is trial 4 with value: 0.780113183484

In [ ]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')